# Understory API Review

## Import and Setup

### Environment Setup

🚨🚨 These steps are required 🚨🚨

1. Run ```cp .env.copy .env```
1. Add your API key to `API_KEY=` in `.env`
1. Also add the domain to `DOMAIN=` for your API, eg `abc-api.understorytech.com`
1. Install dependencies `pip3 install -r requirements.txt`

In [ ]:
import os, re, requests, json
from dotenv import load_dotenv

# Load environment variables
load_dotenv("./.env")

# Setup API configuration
headers = {
    'x-api-key': os.getenv('API_KEY')
}

domain = os.getenv('DOMAIN')
BASE_URL = f"https://{domain}/companies"

print(f"Using BASE_URL: {BASE_URL}")

When we return `4xx` or `5xx` errors, we often also include a helpful message on why the error occured in the message body.
To make this easier to see if something goes wrong with our API usage, we're setting up a utility function to print that message body for us.

In [2]:
def raise_for_status_with_body(response):
    """
    Custom version of raise_for_status that includes response body in error message.
    Args:
        response: requests.Response object
    Raises:
        HTTPError: with detailed error message including response body
    """
    try:
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Status Code: {response.status_code}")
        try:
            error_body = response.json()
            print("Error Response Body:", error_body)
        except ValueError:
            print("Error Response Body:", response.text)
        raise e

## Get Companies

In [ ]:
res = requests.get(BASE_URL, headers=headers)
raise_for_status_with_body(res)
print(res.json())

## Create a Company

In [ ]:
payload = {"companyName": "Apple inc"}
res = requests.post(BASE_URL, headers=headers, json=payload)
raise_for_status_with_body(res)

# Extract the newly created company_id from the response
response_data = res.json()
print(response_data)
company_id = response_data["companyId"]
print(f"Created company with ID: {company_id}")

web_url = re.sub(r'-api(?=\.)', '', domain)
print("🚨 Go to this URL if you want to view your created company in our UI")
print(f"https://{web_url}/company/{company_id}")

## Get Company Files

In [ ]:
res = requests.get(f"{BASE_URL}/{company_id}/files", headers=headers)
raise_for_status_with_body(res)
print(res.json())

## Create New File

In [21]:
# maintain a list of the created file ids, so that we can provide them when generating a model later
created_file_ids = []

In [ ]:
payload = {
    "filename": "Q3-2021.pdf",
    # I picked these pages from our example PDF because they have tables on them
    "pageIngest": [1, 7, [11, 12]]
}

res = requests.post(
    f"{BASE_URL}/{company_id}/files", 
    headers=headers, 
    json=payload
)
raise_for_status_with_body(res)

response_data = res.json()

file_id = response_data["fileId"]
# keep track of the file id so we can use it in model gen later
created_file_ids.append(file_id)
print(f"Created file with ID: {file_id}")

# Get the signed upload url so that we can upload the file directly to s3. 
# NOTE: This signed url has a 900 second expiration.
upload_url = response_data["uploadUrl"]
print(f"Upload URL: {upload_url}")

# We're going to upload our sample PDF file here for testing
with open("AcmeStores 2021 FQ3PR.pdf", 'rb') as file:
    upload_response = requests.put(upload_url, data=file)
    raise_for_status_with_body(upload_response)
    print("File uploaded successfully!")

## Important Context on Files
Now that the file has been uploaded, Understory will do a bunch of processing to extract data from that file. This is different then what we refer to as the model generation process, as it is done on individual files independent of other files. In order for files to be used in the model generation process, they must finish processing (`status=succeeded`). As a convenience, we've made it easy to queue generating a model so you don't have to wait for these file's to finish processing. Immediately after updating your files, you can send a request to generate a model with the `fileId`s you got, and it will automatically wait until those files have finished processing before generating your model for you.

We'll create some better documentation on this soon, but at present these are the current possible statuses for a file:

```
[
  "created",
  "uploaded",
  "pending", // used when re-processing
  "empty", // file is empty
  "importing_started",
  "importing_completed",
  "textract_started",
  "textract_completed",
  "textract_disabled",
  "parsing_started",
  "parsing_completed",
  "failed"
]
```

# Create a Model

In [ ]:
print(f"Creating model with file ids: {created_file_ids}")

payload = {
    "includedFileIds": created_file_ids  # Using the list we've been maintaining
}

res = requests.post(
    f"{BASE_URL}/{company_id}/models",
    headers=headers,
    json=payload
)
raise_for_status_with_body(res)
print(res.json())

model_id = res.json()["modelId"]
print(f"Model created with ID: {model_id}")

Now that our model is being created, we need to wait for it to finish to get the generated result. This process can take anywhere from 2-40 minutes based on how much input data (the files, pages, & tables) were given. This can take significant time because this is the process where we do all the 'stitching' (ie: the process of combining all the tables and timeseries across different files) as well as a bunch of other logic in order to derive a clean model.

For now, the best way to check on the status is to simply long-poll the `GET companies/{companyId}/models/{modelId}` to see the status. In the future, we'll provide a callback URL that will send you a notification once it's done.

# Get a Model

In [ ]:
res = requests.get(
    f"{BASE_URL}/{company_id}/models/{model_id}",
    headers=headers
)
raise_for_status_with_body(res)
model_json = res.json()
print(model_json)

if model_json["status"] == "completed":
    print("Model completed! We can download it now")


## Important context on Models

Like files, models also have statuses. As aforementioned, we'll provide better documentation on this soon. These are the current possible statuses for a model:

```
[
  "created",
  "preparing",
  "running",
  "joining",
  "organizing",
  "summarizing",
  "printing",
  "completed",
  "failed",
] 
```

# Download a Model

Once the model is ready, we can now download it by getting a signed GET url which we can then use to directly download. Note that if the model isn't ready yet, it will return a `400` error.

In [ ]:
res = requests.post(
    f"{BASE_URL}/{company_id}/models/{model_id}/download",
    headers=headers
)
raise_for_status_with_body(res)

model_download_res = res.json()

# NOTE: there's a bug right now where the `modelType` of this response will say "excel", but this is wrong.
print(model_download_res)

download_url = model_download_res["downloadUrl"]
download_response = requests.get(download_url)
raise_for_status_with_body(download_response)

# Parse the JSON content
model_data = download_response.json()

# Option 1: Save the JSON to a file
with open("model_output.json", "w") as f:
    json.dump(model_data, f, indent=2)
print("Model saved to model_output.json")